In [ ]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model # langchain 은 langgraph의 자매 프로젝트 같은 것, 단지 ai 모델이랑 쉽게 대화할 수 잇게 해주는
from langgraph.graph.message import MessagesState
# ToolNode: tools를 호출하는 역할 / tools_condition:  state에서 messages를 꺼내서 거기에 tool call이 있는지 감지하는 역할을 하고 그 조건이 folw(작업흐름)를 tool 노드로 보내줌 
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt # interrupt:  그래프 중단 기능 

llm = init_chat_model("openai:gpt-4o-mini") 

conn = sqlite3.connect(
    "memory.db",
    check_same_thread=False,
)

config = {
    "configurable" : {
        "thread_id" : "2"
    }
}


In [9]:
class State(MessagesState):
    custom_stuff: str


graph_builder = StateGraph(State)

In [ ]:
@tool 
def get_human_feedback(poem:str):   
    """
    Asks the user for feedback on the poem.
    Use this before returning the final response.
    """ 
    feedback = interrupt(f"Here is the poem. tell me what you think\n{poem}")
    #return feedback['feedback'] # interrupt 함수 호출 (input 과 유사 ),아래가 resume={'feedback' : "It looks good!"} 이라면 이렇게 
    return feedback 
    
 
llm_with_tools = llm.bind_tools(tools=[get_human_feedback]) # langchain임 , tools 존재 알려주기 

def chatbot(state: State):
    response = llm_with_tools.invoke(f"""
        Yout are an expert in making poems. 

        Use the `get_human_feedback` tool to get feedback on your poem.

        Only after you receive positive feedback you can return the final poem

        ALWAYS ASK FOR FEEDBACK FIRST.

        HEre is the conversation history

        {state["messages"]}
    """)
    return {"messages": [response]}

In [11]:
tool_node = ToolNode(
    tools=[get_human_feedback],
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition) # tools_condition이 "chatbot -> tools"로 보내거나 "chatbot -> end로 보냄"
graph_builder.add_edge("tools","chatbot") # tools_condition이 tools로 보낸다면 tool을 실행하고 output을 기록함 -> chatbot으로 돌아가서 ai에게 "이게 내가 얻은 tool output이야" 라고 알려주고 -> ai 다시 응답 ->  response로 다시 tools_condition돌아감 -> 더이상 tool calling이 없음을 확인 후 end로 

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn)
)

In [12]:
result = graph.invoke(
    {
        "messages" : [
            {"role" : "user", "content" : "Please make a poem about Python code."}
        ]
    },
    config = config 
)


In [13]:
for message in result["messages"]:
    message.pretty_print() # messages가 langchain 안에서 가지고 있는 메서드인데 메시지를 멋지게 보여준다 .

================================ Human Message =================================

Please make a poem about Python code.
================================== Ai Message ==================================
Tool Calls:
  get_human_feedback (call_zSYHqbYXBvjj8sOpTo1Z7wPM)
 Call ID: call_zSYHqbYXBvjj8sOpTo1Z7wPM
  Args:
    poem: In a realm of logic, clear and bright,
Where syntax dances in the light,
Python, a language, simple yet grand,
With elegant code, we take a stand.

Indentations guide us, like a soft embrace,
Each line a step in a digital space.
From lists to loops, with ease we glide,
In the heart of a function, our dreams reside.

Variables flow like rivers wide,
With data types as friends at our side.
Strings and integers, in harmony blend,
In Python's embrace, we find our trend.

From zero to one, a journey unfolds,
In scripts and modules, our story is told.
With every command, a world we create,
In Python, the coder's joy is innate.

So let us write, with passion and pride,
In a 

In [ ]:
# 그래프의 현재 상태를 볼 수잇다. 
snapshot = graph.get_state(config)

snapshot.interrupts  # interrupt 에 입력한 게 그대로 interrupt에서 확인된다. 

snapshot.next  #다음에 시행될 노드 

(Interrupt(value="Here is the poem. tell me what you think\nIn a realm of logic, clear and bright,\nWhere syntax dances in the light,\nPython, a language, simple yet grand,\nWith elegant code, we take a stand.\n\nIndentations guide us, like a soft embrace,\nEach line a step in a digital space.\nFrom lists to loops, with ease we glide,\nIn the heart of a function, our dreams reside.\n\nVariables flow like rivers wide,\nWith data types as friends at our side.\nStrings and integers, in harmony blend,\nIn Python's embrace, we find our trend.\n\nFrom zero to one, a journey unfolds,\nIn scripts and modules, our story is told.\nWith every command, a world we create,\nIn Python, the coder's joy is innate.\n\nSo let us write, with passion and pride,\nIn a language where creativity cannot hide.\nFor every keystroke, a spark ignites,\nIn the world of Python, our future delights.", id='8ab0037df105036d4589e2484962bdd1'),)

In [15]:
from langgraph.types import Command # Command: 기본적으로 god 모드를 열어줌 또한 그래프를 다시 이어가는데도 유용하다(interrupt를 다시 이어가기 위해 )

response = Command(
    # resume={'feedback' : "It looks good!"} # 'feedback'  부분의 이름은 아무거나 상관없는데 딕셔너리를 ㅇ넣으면 값도 딕셔너리어야함 (위에 feedback 부분! )
    resume = "It looks good!"
)

result = graph.invoke(
    response,  # state 대신에 command 를 전달한다. 대신 command에서는 resume이 잇어야 함
    config = config 
)
for message in result["messages"]:
    message.pretty_print() # messages가 langchain 안에서 가지고 있는 메서드인데 메시지를 멋지게 보여준다 .

================================ Human Message =================================

Please make a poem about Python code.
================================== Ai Message ==================================
Tool Calls:
  get_human_feedback (call_zSYHqbYXBvjj8sOpTo1Z7wPM)
 Call ID: call_zSYHqbYXBvjj8sOpTo1Z7wPM
  Args:
    poem: In a realm of logic, clear and bright,
Where syntax dances in the light,
Python, a language, simple yet grand,
With elegant code, we take a stand.

Indentations guide us, like a soft embrace,
Each line a step in a digital space.
From lists to loops, with ease we glide,
In the heart of a function, our dreams reside.

Variables flow like rivers wide,
With data types as friends at our side.
Strings and integers, in harmony blend,
In Python's embrace, we find our trend.

From zero to one, a journey unfolds,
In scripts and modules, our story is told.
With every command, a world we create,
In Python, the coder's joy is innate.

So let us write, with passion and pride,
In a 